# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
from typing import cast
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)

### data management
import numpy as np
import cv2 
import random

### regression
from sklearn.cluster import MeanShift, estimate_bandwidth

### graphical matplotlib basics
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
# for jupyter notebook management
%matplotlib inline

## 1.2 General dataframe functions

In [ ]:
import smartcheck.paths as pth

## 1.3 General classification functions

In [ ]:
def get_clusters_centroids(X, quantile, n_samples):
    bandwidth = estimate_bandwidth(X, quantile = quantile, n_samples = n_samples)
    cluster = MeanShift(bandwidth = bandwidth)
    cluster.fit(X)
    labels = cluster.labels_
    centroids = cluster.cluster_centers_
    return centroids, labels

# 2. Loading and Data Quality

## 2.1 Loading of data sets and general exploration

#### Using matplotlib

In [ ]:
# Chargement
bird_img_path = pth.get_full_path("smartcheck\\resources\\learning\\bird_small.png")
print("File Full Path:",bird_img_path)
bird_img = plt.imread(bird_img_path)

In [ ]:
# Visualisation
print("Dimensions de l'image:", bird_img.shape)
plt.imshow(bird_img)
plt.xticks([])
plt.yticks([])
plt.show()

#### Using cv2

In [ ]:
# Chargement
bird_img_cv2_path = pth.get_full_path("smartcheck\\resources\\learning\\bird_small.png")
print("File Full Path:",bird_img_cv2_path)
bird_img_cv2_color_BGR = cv2.imread(bird_img_cv2_path, cv2.IMREAD_COLOR)
bird_img_cv2_color_RGB = cv2.cvtColor(bird_img_cv2_color_BGR, cv2.COLOR_BGR2RGB)
bird_img_cv2_gray = cv2.imread(bird_img_cv2_path, cv2.IMREAD_GRAYSCALE)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 8))  # 1 ligne, 2 colonnes
# Image couleur
axs[0].imshow(bird_img_cv2_color_RGB)
axs[0].set_xticks([])
axs[0].set_yticks([])
axs[0].set_title("Image couleur")
# Image en niveaux de gris
axs[1].imshow(bird_img_cv2_gray, cmap='gray')
axs[1].set_xticks([])
axs[1].set_yticks([])
axs[1].set_title("Image en niveaux de gris")
plt.tight_layout()
plt.show()

#### Masking

In [ ]:
# récupération des dimensions de l'image
height, width = bird_img_cv2_gray.shape
# définition des sommet de la formes
sommets = [
    (0, height),
    (width / 2, height / 2),
    (width, height),
]
# transformation en np_array et valeurs entières
sommets = np.array(sommets, np.int32)
# creation d'un masque similaire à la taille de l'image
mask = np.zeros_like(bird_img_cv2_gray)        
# remplissage de ce masque selon une forme géométrique à n sommets avec couleurs full (blanc)
mask = cv2.fillPoly(img=mask, pts=[sommets], color=(255,))
# application du masque à l'image (intersection avec ET logique bit à bit)
intersect_image = cv2.bitwise_and(bird_img_cv2_gray, mask)
# application du masque de l'image (union avec OU logique bit à bit)
union_image = cv2.bitwise_or(bird_img_cv2_gray, mask)
# affichage
fig, axs = plt.subplots(1, 3, figsize=(10, 10))  # 1 ligne, 3 colonnes
# Image couleur
axs[0].imshow(mask, cmap='gray')
axs[0].set_xticks([])
axs[0].set_yticks([])
axs[0].set_title("Masque")
# Image couleur
axs[1].imshow(intersect_image, cmap='gray')
axs[1].set_xticks([])
axs[1].set_yticks([])
axs[1].set_title("Intersection Masque/Image")
# Image en niveaux de gris
axs[2].imshow(union_image, cmap='gray')
axs[2].set_xticks([])
axs[2].set_yticks([])
axs[2].set_title("Union Masque/Image")
plt.tight_layout()
plt.show()

#### Resizing

In [ ]:
bird_img_cv2_color_RGB_downsized = cv2.resize(bird_img_cv2_color_RGB, dsize = (64,64), interpolation = cv2.INTER_AREA)
bird_img_cv2_color_RGB_upsized = cv2.resize(bird_img_cv2_color_RGB, dsize = (256,256), interpolation = cv2.INTER_LINEAR)

fig, axs = plt.subplots(3, 1, figsize=(20, 20))  # 3 lignes, 1 colonne
# Image originale
axs[0].imshow(bird_img_cv2_color_RGB_downsized)
axs[0].set_xticks([])
axs[0].set_yticks([])
axs[0].set_title("Image downsized 64x64")
# Image downsized
axs[1].imshow(bird_img_cv2_color_RGB)
axs[1].set_xticks([])
axs[1].set_yticks([])
axs[1].set_title("Image d'origine 128x128")
# Image downsized
axs[2].imshow(bird_img_cv2_color_RGB_upsized)
axs[2].set_xticks([])
axs[2].set_yticks([])
axs[2].set_title("Image upsized 512x512")

plt.tight_layout()
plt.show()

#### Rotation with scaling or not

In [ ]:
def rotation(img, angle):
    hauteur, largeur = img.shape[:2]
    M = cv2.getRotationMatrix2D((int(hauteur/2), int(largeur/2)), angle, 1)
    img = cv2.warpAffine(img, M, (hauteur, largeur))
    return img  
fig, axs = plt.subplots(1, 2, figsize=(10, 10))  # 1 ligne, 2 colonnes
# Image originale
axs[0].imshow(bird_img_cv2_color_RGB)
axs[0].set_xticks([])
axs[0].set_yticks([])
axs[0].set_title("Image d'origine")
# Image pivotée
axs[1].imshow(rotation(bird_img_cv2_color_RGB, 90))
axs[1].set_xticks([])
axs[1].set_yticks([])
axs[1].set_title("Image pivotée +90° (sens positif = anti-horaire)")
plt.tight_layout()
plt.show()

#### Flip (Mirror)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 10))  # 1 ligne, 2 colonnes
# Image originale
axs[0].imshow(bird_img_cv2_color_RGB)
axs[0].set_xticks([])
axs[0].set_yticks([])
axs[0].set_title("Image d'origine")
# Image pivotée
axs[1].imshow(cv2.flip(bird_img_cv2_color_RGB, 0))
axs[1].set_xticks([])
axs[1].set_yticks([])
axs[1].set_title("Image miroir (flip)")
plt.tight_layout()
plt.show()

#### Zoom

In [ ]:
def zoom(img, coord: np.array):
    if not isinstance(coord, np.ndarray) or coord.shape != (2, 2):
        return img
    else:
        larg_deb, haut_deb = sorted(coord[0])
        larg_fin, haut_fin = sorted(coord[1])
        hauteur, largeur = img.shape[:2]
        larg_deb = max(0, larg_deb)
        larg_fin = min(larg_fin, largeur)
        haut_deb = max(0, haut_deb)
        haut_fin = min(haut_fin, hauteur)
        img_zoom = img[haut_deb:haut_fin, larg_deb:larg_fin, :]
        return img_zoom  
fig, axs = plt.subplots(1, 2, figsize=(10, 10))  # 1 ligne, 2 colonnes
# Image originale
axs[0].imshow(bird_img_cv2_color_RGB)
axs[0].set_xticks([])
axs[0].set_yticks([])
axs[0].set_title("Image d'origine")
# Image pivotée
coord = np.array([[25,25], [75,75]]) # coordonée début(Xd,Yd) et fin(Xf,Yf) du zoom référence 0,0 en haut a gauche
axs[1].imshow(zoom(bird_img_cv2_color_RGB, coord))
axs[1].set_xticks([])
axs[1].set_yticks([])
axs[1].set_title("Zoom image")
plt.tight_layout()
plt.show()

#### Noyau de convolution personnalisée / "blur" / "medianBlur" / "GaussianBlur" et Filtre de niveau adaptatif

In [ ]:
# dégradation poivre et sel
def degrade_poivre_et_sel(img, salt_value=20):
    '''
    Retourne une image corrompue avec un taux de sel et poivre égal à 2/(salt_value+1)%
    NB : avec salt_value = 1 : bruit blanc (100% de sel et poivre)
    '''
    hauteur, largeur = img.shape[:2]
    noise = np.random.randint(salt_value+1, size=(hauteur, largeur))
    img_sp_noise = img.copy()
    # ajout noir random
    indexes = np.where(noise == 0)
    A = indexes[0]
    B = indexes[1]
    img_sp_noise[A,B,:] = 0
    # ajout blanc random
    indexes = np.where(noise == salt_value)
    A = indexes[0]
    B = indexes[1]
    img_sp_noise[A,B,:] = 255
    return img_sp_noise

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12, 18))  # 3 lignes, 2 colonnes
bird_img_cv2_color_RGB_degradee = degrade_poivre_et_sel(bird_img_cv2_color_RGB, 20)
# Image originale
axs[0,0].imshow(bird_img_cv2_color_RGB_degradee)
axs[0,0].set_xticks([])
axs[0,0].set_yticks([])
axs[0,0].set_title("Image d'origine avec bruit sel&poivre")
# Image blurred
axs[1,0].imshow(cv2.blur(bird_img_cv2_color_RGB_degradee, ksize=(3,3)))
axs[1,0].set_xticks([])
axs[1,0].set_yticks([])
axs[1,0].set_title("Image noyau blur 3x3")
# Image avec noyau perso
noyau = np.array([
    [0.7, 0.3, 0.7],
    [0.3, 0.0, 0.3],
    [0.7, 0.3, 0.7]
], dtype=np.float32)
noyau_normalise = noyau / noyau.sum() # la normalisation permet de ne pas altérer globalement la luminosité
axs[0,1].imshow(cv2.filter2D(bird_img_cv2_color_RGB_degradee, ddepth=-1, kernel=noyau_normalise))
axs[0,1].set_xticks([])
axs[0,1].set_yticks([])
axs[0,1].set_title("Image noyau perso 3x3")
# Image avec noyau medianBlur
axs[1,1].imshow(cv2.medianBlur(bird_img_cv2_color_RGB_degradee, ksize=3))
axs[1,1].set_xticks([])
axs[1,1].set_yticks([])
axs[1,1].set_title("Image noyau medianBlur 3x3")
# Image avec noyau GaussianBlur
axs[2,0].imshow(cv2.GaussianBlur(bird_img_cv2_color_RGB_degradee, ksize = (3,3), sigmaX = 0))
axs[2,0].set_xticks([])
axs[2,0].set_yticks([])
axs[2,0].set_title("Image noyau GaussianBlur 3x3 sigmaX 0")
# Image avec filtre de niveau adaptatif
bird_img_cv2_gray_filtree = cv2.adaptiveThreshold(
    bird_img_cv2_gray,
    255,
    adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    thresholdType=cv2.THRESH_BINARY,
    blockSize=5,
    C=3 # négatif : favorise le noir, positif : favorise le blanc
)
axs[2,1].imshow(bird_img_cv2_gray_filtree, cmap='gray')
axs[2,1].set_xticks([])
axs[2,1].set_yticks([])
axs[2,1].set_title("Image filtre adaptatif")
plt.tight_layout()
plt.show()

#### Détection de contours (Canny) et filtrage Sobel et Laplacian

In [ ]:
def project_lines(image, image_contour):
    lines_img = np.zeros(
        (image.shape[0],image.shape[1],3),
        dtype=np.uint8
    )
    lines = cv2.HoughLinesP(
        image_contour,
        rho=1, # resolution pixel (sur faible image 1 est très bien)
        theta=np.pi/180, # angle de restriction de l'alignement (90°)
        threshold=30, # au moins 30 pixels alignés pour définir une ligne
        minLineLength=10, # au moins 10 pixels pour une ligne
        maxLineGap=5 # permet de conecter des segment éloignés de 5 pixels 
    )
    for line in lines:
        for x1, y1, x2, y2 in line:
            cv2.line(
                lines_img, 
                (x1, y1), 
                (x2, y2), 
                color=[255, 0, 0], 
                thickness=1
            )
    image_result = cv2.addWeighted(
        image, 
        0.5, 
        lines_img, 
        1.0, 
        0.0
    )
    return image_result

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(15, 30))  # 4 lignes, 2 colonnes
# Image originale
axs[0,0].imshow(bird_img_cv2_color_RGB)
axs[0,0].set_xticks([])
axs[0,0].set_yticks([])
axs[0,0].set_title("Image d'origine")
# Image contour
bird_img_cv2_color_RGB_blurred = cv2.GaussianBlur(bird_img_cv2_color_RGB,(3,3),0)
bird_img_cv2_color_RGB_contour = cv2.Canny(bird_img_cv2_color_RGB_blurred,100,200)
axs[0,1].imshow(bird_img_cv2_color_RGB_contour, cmap='gray')
axs[0,1].set_xticks([])
axs[0,1].set_yticks([])
axs[0,1].set_title("Image contour apres gaussian blur")
# Image originale avec lignes contour
bird_img_cv2_color_RGB_withlines = project_lines(bird_img_cv2_color_RGB, bird_img_cv2_color_RGB_contour)
axs[1,0].imshow(bird_img_cv2_color_RGB_withlines)
axs[1,0].set_xticks([])
axs[1,0].set_yticks([])
axs[1,0].set_title("Image originale avec lignes contour")
# Image avec filtre Sobel
bird_img_cv2_gray_sobel = cv2.Sobel(bird_img_cv2_gray, ddepth = cv2.CV_64F, dx = 1, dy = 0)
axs[1,1].imshow(bird_img_cv2_gray_sobel, cmap='gray')
axs[1,1].set_xticks([])
axs[1,1].set_yticks([])
axs[1,1].set_title("Image avec filtre Sobel")
# Image avec filtre Laplacian
bird_img_cv2_gray_laplacian = cv2.Laplacian(bird_img_cv2_gray, ddepth = cv2.CV_64F)
axs[2,0].imshow(bird_img_cv2_gray_laplacian, cmap='gray')
axs[2,0].set_xticks([])
axs[2,0].set_yticks([])
axs[2,0].set_title("Image avec filtre Laplacian")
# Image avec érosion 5x5
kernel_erode = np.ones((5,5),np.uint8)
bird_img_cv2_gray_erode = cv2.erode(bird_img_cv2_gray, kernel_erode)
axs[2,1].imshow(bird_img_cv2_gray_erode, cmap='gray')
axs[2,1].set_xticks([])
axs[2,1].set_yticks([])
axs[2,1].set_title("Image érodée 1 fois en 5x5")
# Image avec dilatation 5x5 de l'erosion
kernel_dilate = np.ones((5,5),np.uint8)
bird_img_cv2_gray_dilate = cv2.dilate(bird_img_cv2_gray_erode, kernel_dilate)
axs[3,0].imshow(bird_img_cv2_gray_dilate, cmap='gray')
axs[3,0].set_xticks([])
axs[3,0].set_yticks([])
axs[3,0].set_title("Image dilatée sur l'erosion en 5x5")
# Image différence entre initiale et dilatation
bird_img_diff = bird_img_cv2_gray - bird_img_cv2_gray_erode
axs[3,1].imshow(bird_img_diff, cmap='gray')
axs[3,1].set_xticks([])
axs[3,1].set_yticks([])
axs[3,1].set_title("Image différence initiale-dilatée")
plt.tight_layout()
plt.show()

## 2.2 Data quality refinement

In [ ]:
# Remaniement des dimensions de l'image en 2D 
# suppression de la transparence alpha (RGB Alpha -> RGB) pour diminuer le nombre de variables d'entrainement
bird_img = bird_img[:, :, :3]
bird_rs = np.reshape(bird_img, (bird_img.shape[0]*bird_img.shape[1], bird_img.shape[2]))

In [ ]:
# Visualisation
print("Dimensions de l'image:", bird_img.shape)
plt.imshow(bird_img);

# 3. Data Clustering

## 3.1 General Analysis

In [ ]:
# Visualisation brute des données
# R, G, B pour les axes
r, g, b = bird_rs[:, 0], bird_rs[:, 1], bird_rs[:, 2]
fig = plt.figure(figsize=(10,10))
ax = cast(Axes3D, fig.add_subplot(111, projection='3d'))
ax.scatter(r, g, b, c=bird_rs, marker='o')
ax.view_init(elev=10, azim=90)
ax.set_xlabel('Red')
ax.set_ylabel('Green')
ax.set_zlabel('Blue')
ax.set_title('RGB image Scatter 3D')
plt.show()

## 3.2 Agglomerative Clustering (CAH : Classification Ascendante Hiérarchique )

In [ ]:
# Récupération des information des clusters et visualisation /!\ TRES COUTEUX
centroids, labels = get_clusters_centroids(bird_rs, 0.1, 200)

In [ ]:
# Application de la compression mathématique aux données
print(centroids.shape, centroids)
print(labels.shape, labels)
bird_rs_zip = np.zeros(bird_rs.shape)
for i in range(len(bird_rs_zip)):
    bird_rs_zip[i] = centroids[labels[i]]

In [ ]:
# reconstitution de l'image d'origine
bird_img_zip = np.reshape(bird_rs_zip, (bird_img.shape[0], bird_img.shape[1], bird_img.shape[2]))
plt.figure()
plt.subplot(121)
plt.imshow(bird_img)
plt.title('Image originale')
plt.subplot(122)
plt.imshow(bird_img_zip)
plt.title('Image reconstruite')
plt.show()